# Process Memory Caps and NumPy Allocation

This notebook uses helper functions in `utils.py` to demo simplified swap behavior:
cap this process memory, try too-large NumPy allocation, then restore normal behavior.


## 1. Import the helper module and dependencies

Load the helper file directly from the `.py` file.


In [1]:
from __future__ import annotations

import importlib.util
from pathlib import Path
import sys

import numpy as np
import psutil

helper_path = Path.cwd() / "utils.py"
spec = importlib.util.spec_from_file_location("utils", helper_path)
assert spec is not None and spec.loader is not None
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)

print(f"Loaded helper from: {helper_path}")
print(f"Python platform: {sys.platform}")

Loaded helper from: /Users/alessandrofelder/dev/slides-large-array-data-osss-2026/utils.py
Python platform: darwin


## 2. Inspect system memory and platform

Print RAM size and platform so the demo stays grounded in the current machine.


In [2]:
vm = psutil.virtual_memory()
print(f"Platform: {sys.platform}")
print(f"Total RAM: {vm.total / 1024**3:.2f} GiB")
print(f"Available RAM: {vm.available / 1024**3:.2f} GiB")
print("Expected helper path:")
print("- POSIX: resource.setrlimit(resource.RLIMIT_AS, ...)")
print("- Windows: Job Object with JOB_OBJECT_LIMIT_PROCESS_MEMORY")

Platform: darwin
Total RAM: 16.00 GiB
Available RAM: 9.19 GiB
Expected helper path:
- POSIX: resource.setrlimit(resource.RLIMIT_AS, ...)
- Windows: Job Object with JOB_OBJECT_LIMIT_PROCESS_MEMORY


## 3. Apply a process memory cap

Demo: cap this process to 90% RAM. On macOS fallback is simulated cap mode.


In [3]:
cap_bytes = helper.disable_swap(fraction=0.9)
print(f"Cap mode: {helper.get_cap_mode()}")
print(f"Applied cap: {cap_bytes:,} bytes")
print(f"Applied cap: {cap_bytes / 1024**3:.2f} GiB")

Cap mode: simulated
Applied cap: 15,461,882,265 bytes
Applied cap: 14.40 GiB


## 4. Trigger and observe allocation failure

Demo: allocate via helper so cap behavior is consistent across OS.


In [4]:
target_gib = vm.total / 1024**3 * 2
n_elements = int(target_gib * 1024**3 / np.dtype(np.float64).itemsize)
print(f"Target array size: {target_gib:.2f} GiB")

try:
    arr = helper.random_array_float64(n_elements)
    print(f"Unexpected success: {arr.nbytes / 1024**3:.2f} GiB")
except (MemoryError, OSError) as exc: #, ValueError, RuntimeError) as exc:
    print(f"Allocation failed as expected: {type(exc).__name__}: {exc}")

Target array size: 32.00 GiB
Allocation failed as expected: MemoryError: Simulated cap hit on this platform: requested allocation would exceed process memory cap.


## 5. Restore the original memory settings

Restore normal behavior after the demo.


In [ ]:
helper.enable_swap()
print("Memory cap removed.")

Memory cap removed.


: 

In [ ]:

import time
target_gib = vm.total / 1024**3 / 4
for i in range(5):
    n_elements = int(target_gib * 1024**3 / np.dtype(np.float64).itemsize)
    print(f"Target array size: {target_gib:.2f} GiB")

    try:
        t0 = time.perf_counter()
        arr = helper.random_array_float64(n_elements)
        elapsed_s = time.perf_counter() - t0
        print(f"Allocation of {arr.nbytes / 1024**3:.2f} GiB successful in {elapsed_s:.3f} s")
    except (MemoryError, OSError) as exc: #, ValueError, RuntimeError) as exc:
        print(f"Allocation failed as expected: {type(exc).__name__}: {exc}")

    target_gib *=2

    print(f"Allocation succeeded for array of size {arr.nbytes}")

Target array size: 4.00 GiB
Allocation of 4.00 GiB successful in 2.982 s
Allocation succeeded for array of size 4294967296
Target array size: 8.00 GiB
Allocation of 8.00 GiB successful in 8.424 s
Allocation succeeded for array of size 8589934592
Target array size: 16.00 GiB
Allocation of 16.00 GiB successful in 15.893 s
Allocation succeeded for array of size 17179869184
Target array size: 32.00 GiB
